# 01 - Análise Exploratória de Dados (EDA)
## Lending Club Loan Data

**Tech Challenge - Fase 04 | Machine Learning Engineering**

Este notebook documenta a análise exploratória utilizada para construir a base de um modelo de **Credit Scoring**. O foco não é apenas descrever os dados, mas identificar riscos de qualidade, possíveis fontes de **Data Leakage**, comportamento da variável alvo e características que devem orientar o pré-processamento e o modelo baseline.

### Objetivos

- compreender dimensão, cobertura e estrutura do dataset;
- definir uma variável alvo binária com desfecho de crédito conhecido;
- avaliar valores ausentes e possíveis inconsistências;
- identificar variáveis que não poderiam estar disponíveis no momento da concessão;
- selecionar um conjunto inicial de features para o baseline;
- analisar relações entre características financeiras e inadimplência;
- preservar a dimensão temporal para apoiar análises posteriores de drift.

> **Importante:** este notebook é diagnóstico. Nenhuma observação é removida ou corrigida definitivamente aqui. As transformações ficam para o pipeline de pré-processamento.

## 0. Configuração do ambiente

O dataset completo é grande, portanto algumas análises iniciais são executadas em **chunks**. Essa estratégia permite utilizar toda a base sem carregá-la integralmente em memória com as 151 colunas originais.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, FuncFormatter

In [ ]:
# Exibição
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Paleta visual do notebook
COLORS = {
    "navy": "#17324D",
    "blue": "#377DFF",
    "teal": "#00A6A6",
    "green": "#2CB67D",
    "gold": "#F4B942",
    "coral": "#FF6B6B",
    "purple": "#7B61FF",
    "gray": "#6B7280",
    "light_gray": "#E5E7EB",
}

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# 1. Carregamento e visão geral

In [ ]:
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em: {DATA_PATH.resolve()}\n"
        "Coloque o arquivo accepted_2007_to_2018Q4.csv em data/raw/."
    )

print(f"Dataset localizado em: {DATA_PATH.resolve()}")

In [ ]:
preview = pd.read_csv(DATA_PATH, nrows=5)

print(f"Quantidade de colunas: {preview.shape[1]}")
preview

In [ ]:
preview.columns.tolist()

## 1.1 Leitura do dataset completo em chunks

Nesta primeira varredura, o objetivo é obter estatísticas globais de **volume**, **loan status** e **valores ausentes** sem manter as 151 colunas de mais de 2 milhões de registros simultaneamente em memória.

In [ ]:
CHUNK_SIZE = 100_000

total_rows = 0
loan_status_counts = None
missing_counts = None

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=True
):
    total_rows += len(chunk)

    current_status = chunk["loan_status"].value_counts(dropna=False)
    loan_status_counts = (
        current_status
        if loan_status_counts is None
        else loan_status_counts.add(current_status, fill_value=0)
    )

    current_missing = chunk.isna().sum()
    missing_counts = (
        current_missing
        if missing_counts is None
        else missing_counts.add(current_missing, fill_value=0)
    )

loan_status_counts = loan_status_counts.sort_values(ascending=False).astype(int)

print(f"Total de registros: {total_rows:,}")
print(f"Total de colunas:   {len(preview.columns):,}")

### Visão geral observada

A base completa possui aproximadamente **2,26 milhões de registros e 151 colunas**, o que justifica o processamento inicial em chunks. O volume é muito superior ao mínimo exigido para o desafio e permite trabalhar com uma população ampla sem amostragem na etapa exploratória.

# 2. Variável de resultado: `loan_status`

Antes de definir a variável alvo, é necessário observar todos os possíveis desfechos registrados pelo Lending Club. Status ainda ativos ou intermediários não devem ser rotulados prematuramente como inadimplência ou sucesso.

In [ ]:
loan_status_summary = pd.DataFrame({
    "count": loan_status_counts,
    "percent": loan_status_counts / total_rows * 100
})

loan_status_summary

In [ ]:
status_plot = loan_status_summary.copy()
status_plot.index = status_plot.index.astype(str).str.replace("nan", "NaN", regex=False)
status_plot = status_plot.sort_values("count")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    status_plot.index,
    status_plot["count"],
    color=[
        COLORS["light_gray"] if label == "NaN" else
        COLORS["green"] if "Fully Paid" in label else
        COLORS["coral"] if ("Charged Off" in label or label == "Default") else
        COLORS["blue"]
        for label in status_plot.index
    ]
)

ax.set_title("Distribuição dos status dos empréstimos")
ax.set_xlabel("Quantidade de registros")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1_000_000:.1f}M" if x >= 1_000_000 else f"{x/1_000:.0f}k"))

for bar, value in zip(bars, status_plot["percent"]):
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height()/2,
        f"  {value:.2f}%",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

## 2.1 Definição da variável alvo

Para o problema de Credit Scoring serão utilizados apenas empréstimos com **desfecho financeiro conhecido**.

**Classe 0 - Não inadimplente**
- `Fully Paid`
- `Does not meet the credit policy. Status:Fully Paid`

**Classe 1 - Inadimplente**
- `Charged Off`
- `Default`
- `Does not meet the credit policy. Status:Charged Off`

Status como `Current`, `Late` e `In Grace Period` serão excluídos da população de modelagem porque representam contratos ainda abertos ou com resultado não definitivo. Essa decisão reduz o risco de atribuir um rótulo incorreto a um empréstimo que ainda poderia ser regularizado.

In [ ]:
GOOD_STATUS = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
]

BAD_STATUS = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
]

good_count = int(loan_status_counts.reindex(GOOD_STATUS, fill_value=0).sum())
bad_count = int(loan_status_counts.reindex(BAD_STATUS, fill_value=0).sum())
modeling_count = good_count + bad_count
default_rate = bad_count / modeling_count

target_overview = pd.DataFrame({
    "classe": ["Não inadimplente", "Inadimplente"],
    "registros": [good_count, bad_count],
    "percentual": [good_count / modeling_count * 100, bad_count / modeling_count * 100],
})

print(f"População de modelagem: {modeling_count:,}")
print(f"Taxa de inadimplência:   {default_rate:.2%}")
target_overview

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

bars = ax.bar(
    target_overview["classe"],
    target_overview["percentual"],
    color=[COLORS["teal"], COLORS["coral"]],
    width=0.58,
)

ax.set_title("Distribuição da variável alvo")
ax.set_ylabel("Percentual da população")
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_ylim(0, max(target_overview["percentual"]) * 1.18)

for bar, pct, n in zip(
    bars,
    target_overview["percentual"],
    target_overview["registros"]
):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        f"{pct:.2f}%\n({n:,.0f})",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

### Interpretação

A população final tem aproximadamente **80% de empréstimos não inadimplentes e 20% inadimplentes**. Existe desbalanceamento, porém a classe minoritária ainda contém centenas de milhares de registros. Por isso, nenhuma técnica de balanceamento será aplicada nesta etapa. Na modelagem, o desempenho deverá ser analisado com métricas além de acurácia, como **ROC-AUC, Precision, Recall, F1 e PR-AUC**.

# 3. Qualidade de dados e valores ausentes

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": missing_counts.astype(int),
    "missing_percent": missing_counts / total_rows * 100,
})

missing_summary = (
    missing_summary
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

missing_summary.head(30)

In [ ]:
missing_bins = pd.cut(
    missing_summary["missing_percent"],
    bins=[0, 10, 30, 50, 70, 90, 100],
    include_lowest=True
)

missing_by_band = missing_bins.value_counts().sort_index()
missing_by_band

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))

bars = ax.bar(
    [str(x) for x in missing_by_band.index],
    missing_by_band.values,
    color=[COLORS["teal"], COLORS["blue"], COLORS["purple"], COLORS["gold"], COLORS["coral"], COLORS["navy"]],
)

ax.set_title("Quantidade de colunas por faixa de valores ausentes")
ax.set_xlabel("Percentual de valores ausentes")
ax.set_ylabel("Número de colunas")
ax.tick_params(axis="x", rotation=25)

for bar, value in zip(bars, missing_by_band.values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.8,
        f"{int(value)}",
        ha="center",
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

In [ ]:
top_missing = missing_summary.head(20).sort_values("missing_percent")

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    top_missing.index,
    top_missing["missing_percent"],
    color=COLORS["purple"]
)

ax.set_title("Top 20 variáveis com maior percentual de ausência")
ax.set_xlabel("Valores ausentes (%)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(PercentFormatter(100))

for bar, value in zip(bars, top_missing["missing_percent"]):
    ax.text(
        value + 0.5,
        bar.get_y() + bar.get_height()/2,
        f"{value:.1f}%",
        va="center",
        fontsize=9
    )

ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

### Leitura dos valores ausentes

A alta ausência não deve ser interpretada automaticamente como baixa qualidade. Muitas variáveis `hardship_*`, `settlement_*`, `sec_app_*` e campos conjuntos aparecem apenas em situações específicas.

Por isso, a decisão de exclusão combina **cobertura**, **momento em que a informação se torna disponível** e **utilidade para o cenário de concessão de crédito**.

# 4. Auditoria de variáveis e risco de Data Leakage

Um modelo de concessão deve utilizar apenas informações conhecidas **até o momento da decisão de crédito**. Variáveis que descrevem pagamentos futuros, saldo remanescente, recuperação de valores, dificuldades posteriores ou acordos de dívida podem fornecer ao modelo parte do próprio resultado que ele deveria prever.

Essa situação é tratada como **Data Leakage**.

In [ ]:
feature_audit = pd.DataFrame({"feature": preview.columns})

feature_audit["missing_percent"] = (
    feature_audit["feature"]
    .map(missing_summary["missing_percent"])
    .fillna(0)
)

identifier_cols = [
    "id",
    "member_id",
    "url",
]

post_loan_cols = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "last_credit_pull_d",
]

additional_post_loan = [
    "orig_projected_additional_accrued_interest",
    "payment_plan_start_date",
    "deferral_term",
    "hardship_flag",
    "debt_settlement_flag",
]

post_loan_prefixes = [
    "hardship_",
    "settlement_",
    "debt_settlement_",
]

In [ ]:
def classify_feature(feature):
    if feature == "loan_status":
        return "target"

    if feature in identifier_cols:
        return "identifier"

    if feature == "issue_d":
        return "temporal_key"

    if feature in post_loan_cols or feature in additional_post_loan:
        return "post_loan_leakage"

    if any(feature.startswith(prefix) for prefix in post_loan_prefixes):
        return "post_loan_leakage"

    return "review"


feature_audit["initial_classification"] = (
    feature_audit["feature"].apply(classify_feature)
)

feature_audit["high_missing"] = feature_audit["missing_percent"] >= 80

feature_audit["decision"] = "review"

feature_audit.loc[
    feature_audit["initial_classification"].isin(["identifier", "post_loan_leakage"]),
    "decision"
] = "exclude"

feature_audit.loc[
    feature_audit["initial_classification"] == "target",
    "decision"
] = "target"

feature_audit.loc[
    feature_audit["initial_classification"] == "temporal_key",
    "decision"
] = "keep_metadata"

low_coverage_candidates = [
    "desc",
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
] + [
    col for col in preview.columns if col.startswith("sec_app_")
]

feature_audit.loc[
    feature_audit["feature"].isin(low_coverage_candidates),
    "decision"
] = "exclude_low_coverage"

feature_audit["decision"].value_counts()

In [ ]:
decision_counts = feature_audit["decision"].value_counts()

decision_labels = {
    "review": "Revisão",
    "exclude": "Excluir",
    "exclude_low_coverage": "Baixa cobertura",
    "keep_metadata": "Metadado temporal",
    "target": "Target",
}

decision_colors = {
    "review": COLORS["blue"],
    "exclude": COLORS["coral"],
    "exclude_low_coverage": COLORS["gold"],
    "keep_metadata": COLORS["purple"],
    "target": COLORS["green"],
}

plot_data = decision_counts.sort_values()

fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(
    [decision_labels.get(x, x) for x in plot_data.index],
    plot_data.values,
    color=[decision_colors[x] for x in plot_data.index],
)

ax.set_title("Resultado da auditoria inicial das 151 variáveis")
ax.set_xlabel("Quantidade de variáveis")
ax.set_ylabel("")

for bar, value in zip(bars, plot_data.values):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height()/2,
        f"{value}",
        va="center",
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

In [ ]:
feature_audit.sort_values(
    ["decision", "missing_percent"],
    ascending=[True, False]
)[
    ["feature", "missing_percent", "initial_classification", "decision"]
]

### Principais achados da auditoria

- identificadores técnicos não serão utilizados como preditores;
- variáveis relacionadas a pagamentos, recuperações, hardship e settlement foram classificadas como risco de leakage;
- `issue_d` será preservada como **metadado temporal**, não como feature preditora inicial;
- variáveis de segundo solicitante possuem cobertura muito baixa na população total e não entram no baseline;
- `grade`, `sub_grade` e `int_rate` não são leakage temporal clássico, mas refletem o risco/pricing produzido pelo próprio Lending Club. Para manter o baseline mais independente do score preexistente, essas variáveis não serão usadas na primeira versão.

# 5. Seleção preliminar das features do baseline

O conjunto abaixo prioriza variáveis financeiras, cadastrais e de histórico de crédito plausivelmente conhecidas no momento da concessão.

In [ ]:
baseline_features = [
    "loan_amnt",
    "term",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "pub_rec_bankruptcies",
    "tax_liens",
    "application_type",
]

missing_baseline_cols = [
    col for col in baseline_features if col not in preview.columns
]

if missing_baseline_cols:
    raise ValueError(f"Features não encontradas: {missing_baseline_cols}")

print(f"Features selecionadas para análise do baseline: {len(baseline_features)}")

## 5.1 Construção da população de modelagem

Agora é possível carregar apenas as features selecionadas, reduzindo drasticamente o consumo de memória. Toda a população com desfecho conhecido continua sendo utilizada.

In [ ]:
cols_to_load = baseline_features + ["loan_status", "issue_d"]

chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=cols_to_load,
    chunksize=100_000,
    low_memory=False,
):
    chunk = chunk[chunk["loan_status"].isin(GOOD_STATUS + BAD_STATUS)]
    chunks.append(chunk)

df_modeling = pd.concat(chunks, ignore_index=True)
del chunks

df_modeling["target"] = (
    df_modeling["loan_status"]
    .isin(BAD_STATUS)
    .astype(int)
)

df_modeling["fico_avg"] = (
    df_modeling["fico_range_low"] + df_modeling["fico_range_high"]
) / 2

df_modeling["issue_date"] = pd.to_datetime(
    df_modeling["issue_d"],
    format="%b-%Y",
    errors="coerce"
)

df_modeling["issue_year"] = df_modeling["issue_date"].dt.year

print(f"Shape da população de modelagem: {df_modeling.shape}")
print(f"Taxa de inadimplência: {df_modeling['target'].mean():.2%}")

# 6. Qualidade das features selecionadas

In [ ]:
selected_missing = pd.DataFrame({
    "missing_count": df_modeling[baseline_features].isna().sum(),
    "missing_percent": df_modeling[baseline_features].isna().mean() * 100,
}).sort_values("missing_percent", ascending=False)

selected_missing

In [ ]:
plot_missing = (
    selected_missing[selected_missing["missing_percent"] > 0]
    .sort_values("missing_percent")
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    plot_missing.index,
    plot_missing["missing_percent"],
    color=COLORS["gold"]
)

ax.set_title("Valores ausentes nas features selecionadas")
ax.set_xlabel("Valores ausentes (%)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(PercentFormatter(100))

for bar, value in zip(bars, plot_missing["missing_percent"]):
    ax.text(
        value + 0.08,
        bar.get_y() + bar.get_height()/2,
        f"{value:.2f}%",
        va="center",
        fontsize=9
    )

ax.set_xlim(0, max(plot_missing["missing_percent"].max() * 1.25, 1))
plt.tight_layout()
plt.show()

### Interpretação

A maioria das features selecionadas apresenta cobertura muito alta. `emp_length` se destaca com aproximadamente 5,8% de ausência. Como a própria ausência dessa informação apresenta uma taxa de inadimplência diferente da população geral, ela deve ser preservada como uma categoria explícita no pré-processamento, em vez de ser simplesmente substituída pela moda.

# 7. Distribuições e valores extremos

In [ ]:
numeric_features = (
    df_modeling[baseline_features]
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = [
    col for col in baseline_features if col not in numeric_features
]

print(f"Features numéricas:    {len(numeric_features)}")
print(f"Features categóricas: {len(categorical_features)}")

In [ ]:
numeric_summary = df_modeling[numeric_features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.995, 0.999]
).T

numeric_summary

In [ ]:
outlier_features = [
    "annual_inc",
    "dti",
    "revol_bal",
    "revol_util",
    "open_acc",
    "pub_rec",
    "total_acc",
    "tax_liens",
]

quantile_summary = df_modeling[outlier_features].quantile(
    [0, 0.01, 0.05, 0.50, 0.95, 0.99, 0.995, 0.999, 1]
).T

quantile_summary

In [ ]:
anomaly_checks = pd.Series({
    "dti < 0": (df_modeling["dti"] < 0).sum(),
    "dti > 100": (df_modeling["dti"] > 100).sum(),
    "revol_util > 100": (df_modeling["revol_util"] > 100).sum(),
    "annual_inc <= 0": (df_modeling["annual_inc"] <= 0).sum(),
}, name="count")

anomaly_checks.to_frame()

### Leitura dos extremos

Os valores extremos não serão removidos automaticamente. Alguns podem representar casos raros porém legítimos, como renda muito elevada ou utilização de crédito acima de 100%.

Por outro lado, valores como `dti = -1` indicam possível codificação especial ou inconsistência e deverão ser tratados no pipeline. O pré-processamento deverá distinguir **outlier legítimo** de **valor inválido**.

# 8. Relação entre variáveis numéricas e inadimplência

In [ ]:
comparison_features = [
    "loan_amnt",
    "annual_inc",
    "dti",
    "fico_avg",
    "revol_bal",
    "revol_util",
    "inq_last_6mths",
    "delinq_2yrs",
]

median_comparison = (
    df_modeling
    .groupby("target")[comparison_features]
    .median()
    .T
)

median_comparison.columns = ["Não inadimplente", "Inadimplente"]
median_comparison

In [ ]:
def median_by_target_chart(df, column, title, ylabel, formatter=None):
    medians = df.groupby("target")[column].median().reindex([0, 1])
    labels = ["Não inadimplente", "Inadimplente"]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(
        labels,
        medians.values,
        color=[COLORS["teal"], COLORS["coral"]],
        width=0.58
    )

    ax.set_title(title)
    ax.set_ylabel(ylabel)

    for bar, value in zip(bars, medians.values):
        label = formatter(value) if formatter else f"{value:,.2f}"
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() * 1.02,
            label,
            ha="center",
            va="bottom",
            fontweight="bold"
        )

    ax.set_ylim(0, max(medians.values) * 1.18)
    plt.tight_layout()
    plt.show()

In [ ]:
median_by_target_chart(
    df_modeling,
    "fico_avg",
    "FICO mediano por classe",
    "FICO médio do intervalo",
    formatter=lambda x: f"{x:.0f}",
)

In [ ]:
median_by_target_chart(
    df_modeling,
    "dti",
    "DTI mediano por classe",
    "Debt-to-Income",
    formatter=lambda x: f"{x:.2f}",
)

In [ ]:
median_by_target_chart(
    df_modeling,
    "revol_util",
    "Utilização de crédito rotativo mediana por classe",
    "Revolving utilization (%)",
    formatter=lambda x: f"{x:.1f}%",
)

In [ ]:
# Distribuição central do FICO por classe
# O gráfico utiliza toda a população, apenas restringindo o eixo ao intervalo natural do FICO.
fico_good = df_modeling.loc[df_modeling["target"] == 0, "fico_avg"].dropna()
fico_bad = df_modeling.loc[df_modeling["target"] == 1, "fico_avg"].dropna()

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    fico_good,
    bins=35,
    alpha=0.65,
    density=True,
    color=COLORS["teal"],
    label="Não inadimplente",
)
ax.hist(
    fico_bad,
    bins=35,
    alpha=0.60,
    density=True,
    color=COLORS["coral"],
    label="Inadimplente",
)

ax.set_title("Distribuição do FICO por classe")
ax.set_xlabel("FICO médio")
ax.set_ylabel("Densidade")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### Interpretação

A comparação mostra associações coerentes com risco de crédito: a população inadimplente apresenta **FICO menor**, **DTI maior** e **maior utilização do crédito rotativo**. Essas diferenças não devem ser interpretadas como relações causais, mas indicam potencial utilidade preditiva.

# 9. Variáveis categóricas e taxa de inadimplência

In [ ]:
def default_rate_by_category(df, column):
    result = (
        df
        .groupby(column, dropna=False)["target"]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
    )
    result["default_rate"] = result["mean"] * 100
    return result


def plot_default_rate_category(df, column, title, min_count=0):
    result = default_rate_by_category(df, column).copy()
    result = result[result["count"] >= min_count]

    result.index = [
        "Não informado" if pd.isna(x) else str(x)
        for x in result.index
    ]

    result = result.sort_values("default_rate")

    fig, ax = plt.subplots(figsize=(9, max(4.2, len(result) * 0.42)))
    bars = ax.barh(
        result.index,
        result["default_rate"],
        color=COLORS["blue"]
    )

    ax.set_title(title)
    ax.set_xlabel("Taxa de inadimplência")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(PercentFormatter(100))

    for bar, value in zip(bars, result["default_rate"]):
        ax.text(
            value + 0.25,
            bar.get_y() + bar.get_height()/2,
            f"{value:.1f}%",
            va="center",
            fontsize=9,
        )

    ax.set_xlim(0, max(result["default_rate"].max() * 1.18, 5))
    plt.tight_layout()
    plt.show()

    return result.sort_values("default_rate", ascending=False)

In [ ]:
term_default = plot_default_rate_category(
    df_modeling,
    "term",
    "Taxa de inadimplência por prazo do empréstimo",
)

term_default

In [ ]:
purpose_default = plot_default_rate_category(
    df_modeling,
    "purpose",
    "Taxa de inadimplência por finalidade do empréstimo",
    min_count=300,
)

purpose_default

In [ ]:
home_default = plot_default_rate_category(
    df_modeling,
    "home_ownership",
    "Taxa de inadimplência por situação de moradia",
    min_count=100,
)

home_default

In [ ]:
verification_default = plot_default_rate_category(
    df_modeling,
    "verification_status",
    "Taxa de inadimplência por status de verificação",
)

verification_default

In [ ]:
emp_default = plot_default_rate_category(
    df_modeling,
    "emp_length",
    "Taxa de inadimplência por tempo de emprego",
)

emp_default

In [ ]:
application_default = plot_default_rate_category(
    df_modeling,
    "application_type",
    "Taxa de inadimplência por tipo de aplicação",
)

application_default

### Destaques categóricos

Algumas diferenças são especialmente relevantes para a modelagem:

- contratos de **60 meses** apresentam inadimplência muito superior aos de 36 meses;
- `small_business` aparece entre as finalidades com maior taxa de default;
- clientes em `RENT` apresentam taxa maior que clientes com `MORTGAGE`;
- registros com `emp_length` ausente apresentam comportamento diferente, reforçando a decisão de criar uma categoria específica para ausência;
- `Verified` apresenta inadimplência superior a `Not Verified`, mas isso **não implica causalidade**. O status de verificação pode estar associado ao perfil de risco ou às regras internas de concessão.

# 10. Correlação entre variáveis numéricas

A matriz abaixo é utilizada apenas como diagnóstico de relações lineares. Correlação não substitui análise de importância de features nem implica causalidade.

In [ ]:
corr_features = [
    "loan_amnt",
    "annual_inc",
    "dti",
    "fico_avg",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "target",
]

corr = df_modeling[corr_features].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        value = corr.iloc[i, j]
        ax.text(
            j, i, f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if abs(value) > 0.55 else "black"
        )

ax.set_title("Matriz de correlação das principais variáveis numéricas")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Correlação")

plt.tight_layout()
plt.show()

# 11. Dimensão temporal

`issue_d` representa o período de emissão do empréstimo. Ela será mantida como metadado porque pode apoiar a separação entre conjuntos de referência e produção nas etapas posteriores de detecção de drift.

In [ ]:
year_default = (
    df_modeling
    .groupby("issue_year")["target"]
    .agg(["count", "mean"])
)

year_default["default_rate"] = year_default["mean"] * 100
year_default

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    year_default.index,
    year_default["default_rate"],
    marker="o",
    linewidth=2.5,
    markersize=7,
    color=COLORS["purple"],
)

ax.set_title("Taxa de inadimplência por ano de concessão")
ax.set_xlabel("Ano")
ax.set_ylabel("Taxa de inadimplência")
ax.yaxis.set_major_formatter(PercentFormatter(100))

for year, value in zip(year_default.index, year_default["default_rate"]):
    ax.annotate(
        f"{value:.1f}%",
        (year, value),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8,
    )

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    year_default.index.astype(int).astype(str),
    year_default["count"],
    color=COLORS["teal"],
)

ax.set_title("Volume de empréstimos com desfecho conhecido por ano")
ax.set_xlabel("Ano de concessão")
ax.set_ylabel("Quantidade de empréstimos")
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1_000:.0f}k")
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Cuidado na interpretação temporal:** a população de modelagem inclui apenas empréstimos com desfecho conhecido. Em safras mais recentes, parte dos contratos ainda estava ativa na data de extração e foi excluída. Portanto, a curva anual pode sofrer **viés de maturação** e não deve ser tratada isoladamente como evidência de Concept Drift.

Apesar dessa limitação, a dimensão temporal é valiosa para estruturar cenários posteriores de **Reference Dataset** e **Production Dataset**.

# 12. Principais conclusões do EDA

### População
- O dataset bruto possui aproximadamente **2,26 milhões de registros e 151 colunas**.
- Após considerar apenas empréstimos com desfecho conhecido, a população de modelagem possui **1.348.099 registros**.
- A taxa de inadimplência é próxima de **20%**.

### Qualidade e governança das features
- Grande parte das colunas extremamente esparsas está ligada a cenários específicos ou eventos posteriores à concessão.
- Variáveis pós-empréstimo foram removidas da seleção inicial para evitar **Data Leakage**.
- `issue_d` foi preservada apenas como metadado temporal.
- O baseline não utilizará inicialmente `grade`, `sub_grade` e `int_rate`, reduzindo a dependência do score/pricing já produzido pelo Lending Club.

### Relações com inadimplência
- Inadimplentes apresentam, em média e mediana, **FICO menor**, **DTI maior** e **utilização de crédito rotativo maior**.
- Empréstimos de **60 meses** apresentam taxa de inadimplência substancialmente superior aos de 36 meses.
- A finalidade do empréstimo, situação de moradia, tempo de emprego e tipo de aplicação também apresentam diferenças relevantes entre grupos.

### Dados ausentes e extremos
- As features do baseline possuem boa cobertura geral.
- `emp_length` é a principal variável selecionada com ausência relevante e deverá receber uma categoria explícita de **não informado**.
- Foram encontrados extremos em renda, DTI, utilização rotativa e registros públicos. Eles serão avaliados no pré-processamento sem remoção indiscriminada.

### Próxima etapa
O notebook `02_preprocessing.ipynb` deverá transformar estes achados em um pipeline reproduzível, incluindo:

1. conversão de `term` para valor numérico;
2. tratamento de `emp_length`, preservando a categoria ausente;
3. criação de `fico_avg`;
4. transformação de `earliest_cr_line` em tempo de histórico de crédito;
5. tratamento de valores inválidos e imputação de nulos;
6. codificação de variáveis categóricas;
7. definição de treino, teste e dataset de referência;
8. persistência das transformações para uso consistente em produção.

---
### Nota de reprodutibilidade

Este notebook foi estruturado para utilizar o dataset completo localmente. A pasta `data/` permanece fora do versionamento Git, enquanto o código, análises e resultados do notebook podem ser versionados no repositório.

Antes do commit final, execute **Run All** com o kernel do ambiente `.venv` para que tabelas e gráficos fiquem incorporados ao arquivo `.ipynb`.